# CNN study: preprocessing, length, and capacity

This notebook prepares shared training folds, runs paired raw/normalized experiments,
and confirms finalists. It opens **training data only**. Validation and optional test
evaluation are in `cnn_test.ipynb`.

Start with `SMOKE_MODE = True`: all recipe branches run on a small sample with small
models, one fold, and one epoch. Smoke results cannot establish a winning model.
For the full study, select the GPU-enabled notebook kernel, set smoke mode to
False, and choose a new run name. GPU jobs run sequentially; completed matching
folds resume. No dependencies or drivers are installed by these notebooks.

## 1. Imports and environment

In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "cnn" / "cnn_study.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from cnn import cnn_study as study
from cnn import cnn_experiments as exp

torch.set_num_threads(4)
pd.set_option("display.max_columns", 16)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})

## 2. Choose the run mode

The full study screens 28 entries, adds two combined candidates, and confirms each dataset's reference and best candidate with three seeds. Identical executions reuse their saved results.

In [2]:
SMOKE_MODE = False
RUN_NAME = "cnn_study_v2_full_01"
DEVICE_REQUEST = "cuda:0"

DEVICE = exp.resolve_device(DEVICE_REQUEST)
RUN_DIR = exp.ARTIFACTS_DIR / (f"smoke_{RUN_NAME}" if SMOKE_MODE else RUN_NAME)
print("Mode:", "SMOKE: workflow verification only" if SMOKE_MODE else "FULL STUDY")
print("Run:", RUN_DIR)
display(pd.Series(exp.runtime_info(DEVICE), name="Runtime"))

Mode: FULL STUDY
Run: cnn\artifacts\cnn_study_v2_full_01


python                     3.14.4
torch           2.14.0+rocm10.1.0
numpy                       2.5.4
scikit_learn                1.9.1
device                     cuda:0
device_name     Radeon RX 5500 XT
hip                    7.16.26385
cuda                         None
cpu_threads                     4
Name: Runtime, dtype: object

## 3. Prepare a training-only study

The normalized folder defaults to `data/normalized`, falling back to the existing cleaning export. Original row IDs align datasets. Shared groups include punctuation-removal collisions; labels and rows are retained. No data exports are modified.

In [3]:
settings = study.StudySettings(
    smoke=SMOKE_MODE,
    n_splits=2 if SMOKE_MODE else 5,
    early_stopping_splits=2 if SMOKE_MODE else 10,
)
suite = study.create_suite(RUN_DIR, raw_dir=ROOT / "data", settings=settings)
manifest = suite["manifest"]
display(pd.Series({name: len(ids) for name, ids in manifest["source_rows"].items()},
                  name="Training rows"))
print("Mixed-label transformation-collision groups:", manifest["mixed_label_collision_groups"])
print("Fold assignments are fixed across datasets and training seeds.")

raw           18909
normalized    18830
controlled    18830
Name: Training rows, dtype: int64

Mixed-label transformation-collision groups: 0
Fold assignments are fixed across datasets and training seeds.


## 4. Inspect model and preprocessing recipes

The current reference lowercases and retains punctuation. The six controlled recipes recover original case on the same retained records, separating case and punctuation changes from record removal. Length features always use the original source document before preprocessing and truncation.

In [4]:
model_recipes = pd.DataFrame([{
    "id": c["config_id"], "embedding": c["embedding_dim"], "filters": c["num_filters"],
    "depth": c["depth"], "kernels": c["kernel_sizes"] or [c["kernel_size"]],
    "vocabulary": c["preprocessing"]["max_vocabulary_size"],
    "input_tokens": c["preprocessing"]["max_sequence_length"],
    "length_feature": c["length_feature"],
} for c in manifest["screen_configs"]])
display(model_recipes)
display(pd.DataFrame([{
    "id": c["config_id"], "lowercase": c["preprocessing"]["lowercase"],
    "punctuation": c["preprocessing"]["punctuation_mode"],
} for c in manifest["control_configs"]]))
if SMOKE_MODE:
    print("Smoke execution caps embedding/filter sizes at 8, vocabulary at 512, input at 64.")

,id,embedding,filters,depth,kernels,vocabulary,input_tokens,length_feature
0,reference,64,256,1,[5],20000,1024,none
1,length_tokens,64,256,1,[5],20000,1024,log_tokens
2,length_characters,64,256,1,[5],20000,1024,log_characters
3,vocabulary_40k,64,256,1,[5],40000,1024,none
4,embedding_128,128,256,1,[5],20000,1024,none
5,filters_512,64,512,1,[5],20000,1024,none
6,depth_2,64,256,2,[5],20000,1024,none
7,depth_3,64,256,3,[5],20000,1024,none
8,kernel_7,64,256,1,[7],20000,1024,none
9,parallel_357,64,128,1,"[3, 5, 7]",20000,1024,none


,id,lowercase,punctuation
0,case_0_keep,False,keep
1,case_0_remove_bang_question,False,remove_bang_question
2,case_0_words_only,False,words_only
3,case_1_keep,True,keep
4,case_1_remove_bang_question,True,remove_bang_question
5,case_1_words_only,True,words_only


## 5. Inspect shared folds

Weight-training, early-stopping, and outer held-out groups stay separate. The vocabulary and length statistics are fitted on weight-training rows only. Removing rows for the normalized variant does not move retained records into different partitions.

In [5]:
display(pd.DataFrame([{
    "fold": f["fold"], **{role: len(f[role])
                         for role in ("train", "early_stopping", "heldout")}
} for f in manifest["folds"]]))

,fold,train,early_stopping,heldout
0,1,13614,1513,3782
1,2,13614,1513,3782
2,3,13614,1513,3782
3,4,13614,1513,3782
4,5,13615,1513,3781


## 6. Screen all recipes

This cell performs training. Each shape first exercises real-data Adam updates and records memory and throughput. GPU memory failures retry microbatches 16 and 8 with an effective batch size of 32; unsupported shapes are recorded as failures. There is no silent CPU fallback.

In [6]:
screening = study.run_screening(RUN_DIR, device=DEVICE)
def comparison_table(entries):
    fields = ("dataset", "config_id", "status", "macro_f1_mean",
              "macro_f1_std", "parameter_count")
    return pd.DataFrame([{key: e.get(key) for key in fields} for e in entries])
display(comparison_table(screening))

Resume raw/reference, seed 42, fold 1
Resume raw/reference, seed 42, fold 2
Resume raw/reference, seed 42, fold 3
Resume raw/reference, seed 42, fold 4
Resume raw/reference, seed 42, fold 5
Resume raw/length_tokens, seed 42, fold 1
Resume raw/length_tokens, seed 42, fold 2
Resume raw/length_tokens, seed 42, fold 3
Resume raw/length_tokens, seed 42, fold 4
Resume raw/length_tokens, seed 42, fold 5
Resume raw/length_characters, seed 42, fold 1
Resume raw/length_characters, seed 42, fold 2
Resume raw/length_characters, seed 42, fold 3
Resume raw/length_characters, seed 42, fold 4
Resume raw/length_characters, seed 42, fold 5
Resume raw/vocabulary_40k, seed 42, fold 1
Resume raw/vocabulary_40k, seed 42, fold 2
Resume raw/vocabulary_40k, seed 42, fold 3
Resume raw/vocabulary_40k, seed 42, fold 4
Resume raw/vocabulary_40k, seed 42, fold 5
Resume raw/embedding_128, seed 42, fold 1
Resume raw/embedding_128, seed 42, fold 2
Resume raw/embedding_128, seed 42, fold 3
Resume raw/embedding_128, see

,dataset,config_id,status,macro_f1_mean,macro_f1_std,parameter_count
0,raw,reference,complete,0.979903,0.003319,1362433.0
1,raw,length_tokens,complete,0.979533,0.004120,1362434.0
2,raw,length_characters,complete,0.978158,0.003638,1362434.0
3,raw,vocabulary_40k,complete,0.979215,0.002997,2243636.2
4,raw,embedding_128,complete,0.978528,0.003429,2724353.0
5,raw,filters_512,complete,0.980115,0.003954,1444865.0
6,raw,depth_2,complete,0.975566,0.004359,1690369.0
7,raw,depth_3,complete,0.974562,0.004445,2018305.0
8,raw,kernel_7,complete,0.977840,0.003865,1395201.0
9,raw,parallel_357,complete,0.979110,0.005771,1403649.0


## 7. Fit length-only controls

Both token and character log-length logistic models use the same fold training and held-out records as the CNN. These are diagnostic references, not candidates for automatic CNN selection.

In [7]:
length_baselines = study.run_length_baselines(RUN_DIR)
display(pd.DataFrame([{
    "dataset": r["dataset"], "feature": r["feature"],
    "macro_f1_mean": r["macro_f1_mean"],
} for r in length_baselines]))

,dataset,feature,macro_f1_mean
0,raw,log_tokens,0.691548
1,raw,log_characters,0.663632
2,normalized,log_tokens,0.687213
3,normalized,log_characters,0.661142


## 8. Evaluate combined candidates

For each dataset, combine independently improving vocabulary, embedding, input-length, and length-feature choices. Choose one tested convolution architecture. Add the best eligible controlled preprocessing recipe. The combination is evaluated rather than assumed to improve results.

In [8]:
combined = study.run_combined(RUN_DIR, device=DEVICE)
display(comparison_table(combined))

Train raw/combined, seed 42, fold 1 on cuda:0
Train raw/combined, seed 42, fold 2 on cuda:0
Train raw/combined, seed 42, fold 3 on cuda:0
Train raw/combined, seed 42, fold 4 on cuda:0
Train raw/combined, seed 42, fold 5 on cuda:0
Resume normalized/combined, seed 42, fold 1
Resume normalized/combined, seed 42, fold 2
Resume normalized/combined, seed 42, fold 3
Resume normalized/combined, seed 42, fold 4
Resume normalized/combined, seed 42, fold 5


,dataset,config_id,status,macro_f1_mean,macro_f1_std,parameter_count
0,raw,combined,complete,0.983711,0.002037,1444865.0
1,normalized,combined,complete,0.981412,0.003172,1403649.0


## 9. Confirm references and finalists

Keep split seed 42 and repeat training with seeds 42, 123, and 2026. Initial seed-42 results are reused. Select by mean outer-fold macro F1 at threshold 0.5, then lower standard deviation, fewer parameters, and configuration ID. CV results are model-selection results, not an unbiased final evaluation.

In [9]:
confirmation = study.run_confirmation(RUN_DIR, device=DEVICE)
display(comparison_table(confirmation))

Resume raw/reference, seed 42, fold 1
Resume raw/reference, seed 42, fold 2
Resume raw/reference, seed 42, fold 3
Resume raw/reference, seed 42, fold 4
Resume raw/reference, seed 42, fold 5
Train raw/reference, seed 123, fold 1 on cuda:0
Train raw/reference, seed 123, fold 2 on cuda:0
Train raw/reference, seed 123, fold 3 on cuda:0
Train raw/reference, seed 123, fold 4 on cuda:0
Train raw/reference, seed 123, fold 5 on cuda:0
Train raw/reference, seed 2026, fold 1 on cuda:0
Train raw/reference, seed 2026, fold 2 on cuda:0
Train raw/reference, seed 2026, fold 3 on cuda:0
Train raw/reference, seed 2026, fold 4 on cuda:0
Train raw/reference, seed 2026, fold 5 on cuda:0
Resume raw/combined, seed 42, fold 1
Resume raw/combined, seed 42, fold 2
Resume raw/combined, seed 42, fold 3
Resume raw/combined, seed 42, fold 4
Resume raw/combined, seed 42, fold 5
Train raw/combined, seed 123, fold 1 on cuda:0
Train raw/combined, seed 123, fold 2 on cuda:0
Train raw/combined, seed 123, fold 3 on cuda:0

,dataset,config_id,status,macro_f1_mean,macro_f1_std,parameter_count
0,raw,reference,complete,0.979603,0.003526,1362433.0
1,raw,combined,complete,0.984046,0.002526,1444865.0
2,normalized,reference,complete,0.979410,0.003407,1362433.0
3,normalized,combined,complete,0.981465,0.001898,1403649.0


## 10. Separately retrain final models

Run this cell after reviewing confirmation. Each dataset's selected CNN and reference are trained on all its training rows for the median selected fold epoch count, using seed 42. Vocabulary and length statistics are fitted afresh on those training rows. Validation remains unopened.

In [10]:
selected = study.refit_suite(RUN_DIR, device=DEVICE)
display(pd.DataFrame(selected["selected"]).T)
print("Selected dataset by CV:", selected["overall_selected_dataset"])
print("Next: open cnn_test.ipynb with the same run name and mode.")

Refit raw/combined: 18909 rows, 9 epochs
Refit raw/reference: 18909 rows, 9 epochs
Refit normalized/combined: 18830 rows, 8 epochs
Refit normalized/reference: 18830 rows, 9 epochs


,config_id,macro_f1_mean,macro_f1_std,parameter_count
raw,combined,0.984046,0.002526,1444865.0
normalized,combined,0.981465,0.001898,1403649.0


Selected dataset by CV: raw
Next: open cnn_test.ipynb with the same run name and mode.


## 11. Reproducibility

The run manifest records source hashes, training-file hashes, original row IDs, folds, settings, and runtime versions. Per-fold checkpoints, prediction scores, fitted feature statistics, preflight measurements, and learning histories are saved under the run directory. Changing code, data, or settings requires a new run name.